In [ ]:
import json
import shutil
from pathlib import Path

import anndata as ad
import numpy as np
import os
import pandas as pd

In [ ]:
from tqdm import tqdm

In [ ]:
import pandas as pd

In [ ]:
PLIBDATA_ROOT = '../.plib_cache/raw_datasets'

In [ ]:
cols = ['dataset', 'context', 'perturbation', 'log_dose', 'time']

In [ ]:
SHARDSIZE = 200_000
VALDATA_PORTION = 0.15
TESTDATA_PORTION = 0.15
SPLIT_SEED = 13

In [ ]:
df_annot_all = []
for i, folder in tqdm(enumerate(os.listdir(PLIBDATA_ROOT))):
    for j, file in enumerate(os.listdir(f'{PLIBDATA_ROOT}/{folder}')):
        df_annot = pd.read_parquet(f'{PLIBDATA_ROOT}/{folder}/{file}')[cols].drop_duplicates()
        df_annot_all.append(df_annot)
        
pd.concat(df_annot_all).to_parquet('../.plib_cache/annotations/df_annot_all.parquet', index=False)

In [ ]:
df_annot_all_ = pd.read_parquet('../.plib_cache/annotations/df_annot_all.parquet')

In [ ]:
df_annot_all_

In [ ]:
df_annot_all_['context_perturbation'] = df_annot_all_['context'].astype(str) + '_' + df_annot_all_['perturbation'].astype(str)

In [ ]:
df_annot_all_ = df_annot_all_.drop_duplicates(['dataset','context_perturbation'])

In [ ]:
df_annot_all_

In [ ]:
# step A: per-dataset random train/holdout split. holdout is everything not in train; it will
# be subdivided into val and test in a later cell with a stronger constraint (val and test
# perturbations disjoint from each other).
rng = np.random.RandomState(SPLIT_SEED)
HOLDOUT_PORTION = VALDATA_PORTION + TESTDATA_PORTION

splits = []
for dataset, group in df_annot_all_.groupby('dataset'):
    n = len(group)
    n_holdout = int(n * HOLDOUT_PORTION)
    perm = rng.permutation(n)
    s = np.array(['train'] * n, dtype=object)
    s[perm[:n_holdout]] = 'holdout'
    g = group.copy()
    g['split'] = s
    splits.append(g)

df_annot_split = pd.concat(splits, ignore_index=True)
print(df_annot_split['split'].value_counts())

In [ ]:
# step B: for every holdout pair whose perturbation never appears in train globally, swap it
# with a train pair from the SAME dataset whose perturbation is "multi-dataset safe": the
# perturbation must also appear in train in at least one OTHER dataset (paired with a different
# context), so removing one occurrence here still leaves it in train somewhere via another
# dataset. this preserves per-dataset holdout sizes and keeps every holdout perturbation
# learnable.
rng_sub = np.random.RandomState(SPLIT_SEED + 2000)

train_pert_datasets: dict[str, set[str]] = (
    df_annot_split[df_annot_split['split'] == 'train']
    .groupby('perturbation')['dataset']
    .apply(set).to_dict()
)
def is_multi_ds_safe(pert: str, exclude_dataset: str) -> bool:
    """Pert is in train in at least one dataset other than `exclude_dataset`."""
    ds = train_pert_datasets.get(pert, set())
    return len(ds - {exclude_dataset}) >= 1

holdout_mask = df_annot_split['split'] == 'holdout'
unseen_mask  = holdout_mask & ~df_annot_split['perturbation'].isin(train_pert_datasets.keys())
unseen_idx   = df_annot_split.index[unseen_mask].tolist()

# pre-bucket train candidates by dataset; we filter for multi-ds safety at swap time because
# the safety predicate depends on the exclude_dataset of the unseen pair.
train_by_ds: dict[str, list[int]] = {}
for idx, row in df_annot_split.loc[df_annot_split['split'] == 'train', ['dataset']].iterrows():
    train_by_ds.setdefault(row['dataset'], []).append(idx)
for v in train_by_ds.values():
    rng_sub.shuffle(v)

n_swapped = 0
n_skipped = 0
for idx in unseen_idx:
    target_split = df_annot_split.at[idx, 'split']
    dataset      = df_annot_split.at[idx, 'dataset']
    moved_pert   = df_annot_split.at[idx, 'perturbation']

    pool = train_by_ds.get(dataset, [])
    swap_idx = None
    rejected: list[int] = []
    while pool:
        cand = pool.pop()
        if df_annot_split.at[cand, 'split'] != 'train':
            continue  # already used in a previous swap
        cand_pert = df_annot_split.at[cand, 'perturbation']
        if is_multi_ds_safe(cand_pert, dataset):
            swap_idx = cand
            break
        rejected.append(cand)  # not safe right now; might still be useful later, push back
    pool.extend(rejected)

    if swap_idx is None:
        # no multi-dataset-safe substitute available in the same dataset (e.g. single-context
        # datasets whose perturbations don't overlap with any other dataset). move the unseen
        # pair to train so its embedding is at least learned from this row, and accept the loss
        # of the val/test slot.
        df_annot_split.at[idx, 'split'] = 'train'
        train_pert_datasets.setdefault(moved_pert, set()).add(dataset)
        n_skipped += 1
        continue

    cand_pert = df_annot_split.at[swap_idx, 'perturbation']
    df_annot_split.at[idx, 'split']      = 'train'
    df_annot_split.at[swap_idx, 'split'] = target_split
    # update train_pert_datasets: cand_pert lost a train occurrence in `dataset`; check whether
    # any other train pair of cand_pert remains in `dataset` and update the set accordingly.
    still_in_ds = (
        ((df_annot_split['split'] == 'train')
         & (df_annot_split['dataset'] == dataset)
         & (df_annot_split['perturbation'] == cand_pert)).any()
    )
    if not still_in_ds and cand_pert in train_pert_datasets:
        train_pert_datasets[cand_pert].discard(dataset)
        if not train_pert_datasets[cand_pert]:
            del train_pert_datasets[cand_pert]
    train_pert_datasets.setdefault(moved_pert, set()).add(dataset)
    n_swapped += 1

print(f'unseen-perturbation pairs in val/test: {len(unseen_idx)}')
print(f'  swapped (same-dataset, multi-dataset-safe replacement): {n_swapped}')
print(f'  no replacement available - moved to train instead:      {n_skipped}')
print(df_annot_split['split'].value_counts())

In [ ]:
# step C: split holdout into val/test with perturbations DISJOINT between val and test (across
# all datasets). assignment is at the perturbation level: every unique perturbation currently
# in holdout is randomly assigned to val or test, and all its holdout pairs (across every
# dataset) inherit that label. train pairs are untouched, so the same perturbation is still
# allowed to appear in train (via a different context pair) -- only val vs test sharing is
# forbidden. the val/test pair-count ratio will roughly track VALDATA_PORTION : TESTDATA_PORTION
# but is computed over unique perturbations, so per-dataset val/test pair counts may drift if
# a few perturbations carry many holdout pairs each.
rng_vt = np.random.RandomState(SPLIT_SEED + 5000)

holdout_perts = df_annot_split.loc[df_annot_split['split'] == 'holdout', 'perturbation'].unique()
rng_vt.shuffle(holdout_perts)
n_val_perts = int(round(len(holdout_perts) * VALDATA_PORTION / (VALDATA_PORTION + TESTDATA_PORTION)))
val_perts  = set(holdout_perts[:n_val_perts])
test_perts = set(holdout_perts[n_val_perts:])

mask_val  = (df_annot_split['split'] == 'holdout') & df_annot_split['perturbation'].isin(val_perts)
mask_test = (df_annot_split['split'] == 'holdout') & df_annot_split['perturbation'].isin(test_perts)
df_annot_split.loc[mask_val,  'split'] = 'val'
df_annot_split.loc[mask_test, 'split'] = 'test'

assert (df_annot_split['split'] != 'holdout').all(), 'holdout label still present'
assert val_perts.isdisjoint(test_perts), 'val and test perturbations overlap'
print(f'holdout perturbations split: {len(val_perts)} -> val, {len(test_perts)} -> test')
print(df_annot_split['split'].value_counts())

In [ ]:
# sanity checks on the final split:
#   1) coverage: every perturbation/context in val/test should also appear in train somewhere
#      (so its embedding receives gradient).
#   2) val/test perturbation disjointness: a perturbation must not appear in BOTH val and test.
#      (sharing with train is allowed and necessary for learnability; sharing across val/test
#      would mean the model has seen the pert in val by the time we evaluate on test.)
pert_train = set(df_annot_split.loc[df_annot_split['split'] == 'train', 'perturbation'])
ctx_train  = set(df_annot_split.loc[df_annot_split['split'] == 'train', 'context'])
pert_val   = set(df_annot_split.loc[df_annot_split['split'] == 'val',   'perturbation'])
pert_test  = set(df_annot_split.loc[df_annot_split['split'] == 'test',  'perturbation'])
ctx_holdout = set(df_annot_split.loc[df_annot_split['split'].isin(['val', 'test']), 'context'])

unseen_pert = (pert_val | pert_test) - pert_train
unseen_ctx  = ctx_holdout - ctx_train
val_test_overlap = pert_val & pert_test

print(f'perturbations only in val/test (no train pair anywhere): {len(unseen_pert)}')
print(f'contexts      only in val/test (no train pair anywhere): {len(unseen_ctx)}')
print(f'perturbations shared between val and test:               {len(val_test_overlap)}')

In [ ]:
report = (
    df_annot_split.groupby(['dataset', 'split']).size()
    .unstack(fill_value=0)
    .reindex(columns=['train', 'val', 'test'], fill_value=0)
)
report['total'] = report.sum(axis=1)
for c in ['train', 'val', 'test']:
    report[f'{c}_frac'] = (report[c] / report['total']).round(3)
report

In [ ]:
df_annot_split.columns

In [ ]:
df_annot_split[['dataset', 'context', 'perturbation', 'log_dose', 'time', 'split']].to_parquet(
        '../.plib_cache/annotations/df_annot_split.parquet',
        index=False,
        )